# Exercises XP Ninja ? Intelligent Document Assistant

XP Ninja: Build an Intelligent Document Assistant

## What you'll learn
- Combine foundational and advanced prompt engineering techniques.
- Apply CoT, role prompting, and memory chaining in a real scenario.
- Design a dynamic, multi-step LLM workflow that adapts to user input.
- Mitigate limitations like hallucination and bias.

## What you'll build
- An LLM-powered Document Assistant that: summarizes the contract, answers follow-up questions, keeps context, and self-critiques.

### Document (input)
```
Service Agreement ? Excerpt

This Service Agreement ("Agreement") is made effective as of March 1, 2025, by and between BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").

Scope of Work: Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Exhibit A.

Payment Terms: Client agrees to pay a fixed monthly fee of $12,000, payable within 30 days of receipt of invoice. Late payments will incur a 2% penalty per month.

Term and Termination: This Agreement shall commence on March 1, 2025, and remain in effect for 12 months. Either party may terminate with 30 days? written notice.

Confidentiality: Both parties agree to protect the confidentiality of proprietary or sensitive information shared during the course of the engagement.

Limitation of Liability: Provider?s total liability shall not exceed the fees paid by Client in the 3 months prior to a claim. Provider is not liable for indirect or consequential damages.

Governing Law: This Agreement shall be governed by the laws of the State of California.
```

## Helper: Run a prompt
Uses `ollama run` if available (default model: `llama3`, override with `OLLAMA_MODEL`). If Ollama is missing or fails, prints the prompt in dry-run mode instead of crashing.

In [1]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 200):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None

## Step 1: Initial Summary Prompt
Write a few-shot or Chain-of-Thought prompt to summarize the document in plain English. Include: responsibilities, payment terms, termination, liability.

In [2]:
# TODO: craft a few-shot or CoT summary prompt
summary_prompt = (
    "## Role "
    "You are an intelligent document assistant specialized in summarizing legal contracts for non legal professionals. "
    "## Task "
    "Summarize the contract excerpt below in plain English. "
    "## Reasoning Approach "
    "Identify key sections and explain them clearly without legal jargon. "
    "## Required Content "
    "Include responsibilities of each party payment terms termination conditions and liability limitations. "
    "## Style "
    "Use clear concise language suitable for a general audience. "
    "## Input Document "
    "Service Agreement made effective March 1 2025 between BrightLine Technologies Ltd and NovaWare Systems Inc."
)
summary_prompt


'## Role You are an intelligent document assistant specialized in summarizing legal contracts for non legal professionals. ## Task Summarize the contract excerpt below in plain English. ## Reasoning Approach Identify key sections and explain them clearly without legal jargon. ## Required Content Include responsibilities of each party payment terms termination conditions and liability limitations. ## Style Use clear concise language suitable for a general audience. ## Input Document Service Agreement made effective March 1 2025 between BrightLine Technologies Ltd and NovaWare Systems Inc.'

In [3]:
# Optional: test
run_prompt(summary_prompt)

[dry-run] ollama not installed. Prompt to send: ## Role You are an intelligent document assistant specialized in summarizing legal contracts for non legal professionals. ## Task Summarize the contract excerpt below in plain English. ## Reasoning Approach Identify key sections and explain them clearly without legal jargon. ## Required Content Include responsibilities of each party payment terms termination conditions and liability limitations. ## Style Use clear concise language suitable for a general audience. ## Input Document Service Agreement made effective March 1 2025 between BrightLine Technologies Ltd and NovaWare Systems Inc.


## Step 2: Role-Based Follow-Up Q&A
Role: contract lawyer. Use Narrative-of-Thought or Instance-Adaptive CoT to answer user questions about the agreement.

In [4]:
# TODO: write the role-based Q&A prompt
qa_prompt = (
    "## Role "
    "You are an experienced contract lawyer explaining agreements to business clients. "
    "## Task "
    "Answer user questions about the contract using the document summary and original text. "
    "## Reasoning Style "
    "Use narrative reasoning to explain clauses clearly and logically. "
    "## Constraints "
    "Do not introduce legal interpretations that are not explicitly supported by the document. "
    "If a clause is unclear state that explicitly. "
    "## Example Question "
    "Explain the limitation of liability clause."
)
qa_prompt


'## Role You are an experienced contract lawyer explaining agreements to business clients. ## Task Answer user questions about the contract using the document summary and original text. ## Reasoning Style Use narrative reasoning to explain clauses clearly and logically. ## Constraints Do not introduce legal interpretations that are not explicitly supported by the document. If a clause is unclear state that explicitly. ## Example Question Explain the limitation of liability clause.'

## Step 3: Memory Integration
Simulate context chaining so the assistant remembers the summary and answers accordingly. Choose a method: prior message passing, structured history, or describe vector store retrieval.

In [5]:
# TODO: design the memory/context structure
memory_plan = (
    "Structured conversation history will be used as memory. "
    "The assistant stores a summarized version of the contract and references it when answering follow up questions. "
    "Each user question is answered using both the stored summary and the original document text."
)
memory_plan


'Structured conversation history will be used as memory. The assistant stores a summarized version of the contract and references it when answering follow up questions. Each user question is answered using both the stored summary and the original document text.'

In [6]:
# TODO: sample prompt that injects summary + history
memory_prompt = (
    "## Role "
    "You are an intelligent document assistant with access to prior conversation context. "
    "## Stored Context "
    "You have already generated a summary of the Service Agreement including responsibilities payment terms termination and liability limits. "
    "## Task "
    "Use the stored summary and the original contract text to answer the user question accurately. "
    "## Rules "
    "Maintain consistency with previous answers and do not contradict the stored summary."
)
memory_prompt


'## Role You are an intelligent document assistant with access to prior conversation context. ## Stored Context You have already generated a summary of the Service Agreement including responsibilities payment terms termination and liability limits. ## Task Use the stored summary and the original contract text to answer the user question accurately. ## Rules Maintain consistency with previous answers and do not contradict the stored summary.'

## Step 4: Mitigation & Refinement
Add a self-reflection or multi-agent critique step to review answers for accuracy and clarity.

In [7]:
# TODO: write self-critique / refinement prompt
critique_prompt = (
    "## Role "
    "You are a legal quality reviewer. "
    "## Task "
    "Review the previous answer for factual accuracy clarity and consistency with the contract text. "
    "## Checklist "
    "Verify that no legal claims were invented. "
    "Ensure the explanation matches the contract wording. "
    "Confirm the language is clear and understandable. "
    "## Output "
    "If issues are found explain them briefly and suggest corrections."
)
critique_prompt


'## Role You are a legal quality reviewer. ## Task Review the previous answer for factual accuracy clarity and consistency with the contract text. ## Checklist Verify that no legal claims were invented. Ensure the explanation matches the contract wording. Confirm the language is clear and understandable. ## Output If issues are found explain them briefly and suggest corrections.'

In [8]:
# Optional: chain answer + critique
combined_prompt = (
    "## Step One "
    "Answer the user question about the contract using the stored summary and document text. "
    "## Step Two "
    "Immediately review your answer using a legal quality check for accuracy and clarity. "
    "## Output "
    "Provide the final refined answer after self review."
)
combined_prompt


'## Step One Answer the user question about the contract using the stored summary and document text. ## Step Two Immediately review your answer using a legal quality check for accuracy and clarity. ## Output Provide the final refined answer after self review.'